# Pipeline : Mapping, SNP Calling and SDPOP for *Laurus novocanariensis*

# 1. Mapping and SNP calling

## 1.0 Before running the WORKFLOW

In [ ]:
# Create the directories
mkdir /home/tbertrand/work/L_novocanariensis
mkdir /home/tbertrand/work/L_novocanariensis/GENOME
mkdir /home/tbertrand/work/L_novocanariensis/INPUT
mkdir /home/tbertrand/work/L_novocanariensis/MAPPING
mkdir /home/tbertrand/work/L_novocanariensis/GENOTYPING
mkdir /home/tbertrand/work/L_novocanariensis/WORKFLOW_MAPPING_FREEBAYES

# Download the fasta_generate_regions.py script
cd /home/tbertrand/work/L_novocanariensis/GENOTYPING
wget https://raw.githubusercontent.com/freebayes/freebayes/master/scripts/fasta_generate_regions.py

# Create the input list in the following format (Tab separated)  
R1.fasta    R2.fasta    Indiv_name

# Exemple: 
/home/tbertrand/work/RNA_data/trimming/LN-T1_Mb_R1_001_val_1.fq.gz  /home/tbertrand/work/RNA_data/trimming/LN-T1_Mb_R2_001_val_2.fq.gz	LN-T1_M
/home/tbertrand/work/RNA_data/trimming/LN-T2_F_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T2_F_R2_001_val_2.fq.gz	LN-T2_F
/home/tbertrand/work/RNA_data/trimming/LN-T3_F_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T3_F_R2_001_val_2.fq.gz	LN-T3_F
/home/tbertrand/work/RNA_data/trimming/LN-T3_Female_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T3_Female_R2_001_val_2.fq.gz	LN-T3_F
/home/tbertrand/work/RNA_data/trimming/LN-T4_M_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T4_M_R2_001_val_2.fq.gz	LN-T4_M

# Save it in the INPUT directory
"/home/tbertrand/work/L_novocanariensis/INPUT/reads_list_novocanariensis.txt"

##  1.1 Indexing the reference genome with STAR

In [ ]:
#!/bin/bash
#SBATCH --job-name=index
#SBATCH -p unlimitq
#SBATCH --time=7-00:00:00
#SBATCH --mail-user=tristan.bertrand@umontpellier.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=24

#############################
# 1. MODULES
#############################

module load bioinfo/STAR/2.7.11b

#############################
# 2. FILES 
#############################

WORKDIR=/home/tbertrand/work/L_azorica/L_azorica_L_azorica/GENOME/Index
FASTA=/home/tbertrand/work/L_azorica/L_azorica_L_azorica/GENOME/GCA_977007225.1_dmLauAzor1.pri_genomic.fna
ANNOTATION=/home/tbertrand/work/L_azorica/L_azorica_L_azorica/GENOME/braker.fixed_ids.gff3

#############################
# 3. RUN STAR genomeGenerate
#############################

STAR --runMode genomeGenerate --genomeDir ${WORKDIR}  --genomeFastaFiles ${FASTA}  --runThreadN 24 --sjdbGTFfile ${ANNOTATION}  --sjdbGTFtagExonParentTranscript Parent

## 1.2 Snakemake pipeline

#### config file (config.yaml)

In [ ]:
genome_dir: "/home/tbertrand/work/L_novocanariensis/GENOME/Index"
ref_genome: "/home/tbertrand/work/L_novocanariensis/GENOME/Index/GCA_977007225.1_dmLauAzor1.pri_genomic.fna"
sample_table: "/home/tbertrand/work/L_novocanariensis/INPUT/reads_list_novocanariensis.txt"
mapping_dir: "/home/tbertrand/work/L_novocanariensis/MAPPING"
genotyping_dir: "/home/tbertrand/work/L_novocanariensis/GENOTYPING"

#### SNAKEFILE_Freebayes_V2

In [ ]:
configfile: "config.yaml"

# A METTRE APRES AVOIR FAIT wc -l genotyping_dir/regions/all_regions.bed
NREGIONS = 14015
regions = list(range(1, NREGIONS + 1))

# Lire table samples
samples = {}
with open(config["sample_table"]) as f:
    for line in f:
        r1, r2, ind = line.strip().split()
        samples[ind] = {"r1": r1, "r2": r2}

INDIVIDUALS = list(samples.keys())


rule all:
    input:
        # tous les BAM traités pour freebayes
        expand(config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam", ind=INDIVIDUALS),
        # le fichier BAM_LIST.txt pour freebayes 
        config["genotyping_dir"] + "/BAM_LIST.txt",
        # Tous les VCF chunks générés (pour que Snakemake relance les manquants)
        expand(config["genotyping_dir"] + "/vcf_chunks/variants.{i}.vcf", i=regions),
        # VCF final concaténé
        config["genotyping_dir"] + "L_novocanariensis_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"
        


rule GenerateFreebayesRegions:
    input:
        index=config["ref_genome"] + ".fai"
    output:
        expand(config["genotyping_dir"] + "/regions/region.{i}.bed", i=regions)
    shell:
        """
        module load bioinfo/freebayes/1.3.6
        module load devel/Miniforge/Miniforge3
        module load bioinfo/vcflib/1.0.12
        mkdir -p {config[genotyping_dir]}/regions

        python {config[genotyping_dir]}/fasta_generate_regions.py \
        {input.index} 100000 \
        > {config[genotyping_dir]}/regions/all_regions.bed

        awk '{{ 
            gsub(":", "\\t"); 
            gsub("-", "\\t"); 
            print > "{config[genotyping_dir]}/regions/region." NR ".bed" 
        }}' {config[genotyping_dir]}/regions/all_regions.bed
        """

rule VariantCallingFreebayes:
    input:
        bams=expand(config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam", ind=INDIVIDUALS),
        index=expand(config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam.bai", ind=INDIVIDUALS),
        ref=config["ref_genome"],
        samples=config["genotyping_dir"] + "/BAM_LIST.txt",
        regions=config["genotyping_dir"] + "/regions/region.{i}.bed"
    output:
        temp(config["genotyping_dir"] + "/vcf_chunks/variants.{i}.vcf")
    log:
        config["genotyping_dir"] + "/logs/freebayes_region.{i}.log"
    threads: 1
    resources:
        mem_mb=10000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/freebayes/1.3.6
        module load devel/Miniforge/Miniforge3
        module load bioinfo/vcflib/1.0.12

        mkdir -p {config[genotyping_dir]}/vcf_chunks

        freebayes \
          -f {input.ref} \
          -t {input.regions} \
          -L {input.samples} \
          -C 2 \
          --report-monomorphic \
          --use-best-n-alleles 2 \
          > {output} 2> {log}
        """


rule ConcatVCFs:
    input:
        calls=expand(config["genotyping_dir"] + "/vcf_chunks/variants.{i}.vcf", i=regions)
    output:
        config["genotyping_dir"] + "L_novocanariensis_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"
    log:
        "logs/ConcatVCFs.log"
    threads: 4
    shell:
        """
        module load bioinfo/Bcftools/1.9

        bcftools concat {input.calls} -Ov -o {output}
        """

#### run_snakefile_V2

In [ ]:
#!/bin/bash
#SBATCH --job-name=Snakemake_L_nobilis_all_freebayes_V2
#SBATCH -p workq
#SBATCH --time=4-00:00:00
#SBATCH --mail-user=tristan.bertrand@umontpellier.fr
#SBATCH --mail-type=ALL
#SBATCH --cpus-per-task=1
#SBATCH --mem=8G
#SBATCH --output=snakemake_%j.log

module load bioinfo/Snakemake/8.20.3

snakemake \
  --snakefile snakefile_freebayes \
  --cores 64 \
  --rerun-incomplete \
  --keep-going \
  --latency-wait 60 \
  --executor slurm \
  --jobs 64 \

## 1.3 Results MAPPING

#### Script statistiques mapping: **stats_mapping.sh**

In [ ]:
#!/bin/bash

output="mapping_summary.csv"
echo "Sample,Reads,Unique,Multi,Too_short,Other" > "$output"

for file in *Log.final.out
do
    sample=$(basename "$file" Log.final.out)

    reads=$(grep "Number of input reads" "$file" | awk '{print $NF}')
    unique=$(grep "Uniquely mapped reads %" "$file" | awk '{print $NF}')
    multi=$(grep "% of reads mapped to multiple loci" "$file" | awk '{print $NF}')
    short=$(grep "% of reads unmapped: too short" "$file" | awk '{print $NF}')
    other=$(grep "% of reads unmapped: other" "$file" | awk '{print $NF}')

    echo "$sample,$reads,$unique,$multi,$short,$other" >> "$output"
done

Results Mapping

In [20]:
import pandas as pd
import matplotlib as mp

# Données
data = """Sample,Reads,Unique,Multi,Too_short,Other
LN-T10_M,65955540,91.03%,3.94%,4.28%,0.47%
LN-T11_M,62680258,91.89%,3.65%,3.60%,0.58%
LN-T12_M,67013269,89.39%,3.96%,5.45%,0.87%
LN-T13_F,56698441,89.92%,4.01%,4.24%,1.48%
LN-T14_M,69915724,91.01%,3.99%,4.26%,0.44%
LN-T15_M,52537066,89.77%,3.55%,4.84%,1.57%
LN-T16_M,45167542,92.23%,3.83%,3.24%,0.45%
LN-T17b_M,76797668,91.60%,3.73%,4.01%,0.41%
LN-T18b_M,75558706,91.77%,3.70%,3.52%,0.77%
LN-T19_M,66779529,90.38%,3.98%,5.03%,0.39%
LN-T1_M,59620507,90.08%,4.19%,4.97%,0.47%
LN-T20_M,62860343,78.40%,3.77%,10.02%,7.31%
LN-T21_M,68047303,89.20%,3.98%,6.06%,0.45%
LN-T22_M,56944283,90.62%,3.92%,4.59%,0.47%
LN-T23_M,36758301,90.51%,3.78%,4.63%,0.72%
LN-T24_M,65603044,89.35%,3.98%,4.33%,2.01%
LN-T25,50258522,91.06%,3.97%,4.29%,0.47%
LN-T26_F,68324796,91.54%,4.12%,3.53%,0.49%
LN-T27_F,81417156,81.20%,3.93%,8.71%,5.72%
LN-T28_F,61725748,89.59%,3.72%,4.41%,1.96%
LN-T2_F,48062093,89.94%,4.41%,4.71%,0.60%
LN-T3_F,100388264,74.21%,3.89%,8.02%,13.32%
LN-T4_M,70028228,91.55%,4.12%,3.63%,0.42%
LN-T5_F,103057344,68.04%,3.77%,12.82%,14.70%
LN-T6_M,67572207,92.26%,3.74%,3.54%,0.23%
LN-T7_M,78038790,92.18%,3.71%,3.63%,0.30%"""

# Charger dans pandas
df = pd.read_csv(pd.io.common.StringIO(data))

# Convertir les % en float
for col in ["Unique", "Multi", "Too_short", "Other"]:
    df[col] = df[col].str.replace('%', '').astype(float)

# Formatage visuel
styled_df = (
    df.style
    .background_gradient(subset=["Unique"], cmap="Greens",low=0.4, high=0.8)
    .background_gradient(subset=["Too_short", "Other"], cmap="Reds",low=0.2, high=0.8)
    .background_gradient(subset=["Multi"], cmap="Reds",low=0.4, high=0.8)
    .format({
        "Unique": "{:.2f}%",
        "Multi": "{:.2f}%",
        "Too_short": "{:.2f}%",
        "Other": "{:.2f}%"
    })
)

styled_df

,Sample,Reads,Unique,Multi,Too_short,Other
0,LN-T10_M,65955540,91.03%,3.94%,4.28%,0.47%
1,LN-T11_M,62680258,91.89%,3.65%,3.60%,0.58%
2,LN-T12_M,67013269,89.39%,3.96%,5.45%,0.87%
3,LN-T13_F,56698441,89.92%,4.01%,4.24%,1.48%
4,LN-T14_M,69915724,91.01%,3.99%,4.26%,0.44%
5,LN-T15_M,52537066,89.77%,3.55%,4.84%,1.57%
6,LN-T16_M,45167542,92.23%,3.83%,3.24%,0.45%
7,LN-T17b_M,76797668,91.60%,3.73%,4.01%,0.41%
8,LN-T18b_M,75558706,91.77%,3.70%,3.52%,0.77%
9,LN-T19_M,66779529,90.38%,3.98%,5.03%,0.39%


## 1.4 Results GENOTYPING

#### Script statitstiques VCF: **stats_VCF.sh**

In [ ]:
#!/bin/bash

#SBATCH --job-name=Stats_VCF
#SBATCH -p workq
#SBATCH --time=1-00:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=4

#############################
# 1. MODULES
#############################

module load devel/python/Python-3.6.3
module load bioinfo/Bcftools/1.9
module load bioinfo/samtools/1.14
module load bioinfo/VCFtools/0.1.16

#############################
# 2. FILES 
#############################

VCF=/home/tbertrand/work/L_novocanariensis/GENOTYPING/STATS/L_novocanariensis_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf
OUTPUTDIR=/home/tbertrand/work/L_novocanariensis/GENOTYPING/STATS
BASENAME=L_novocanariensis_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2

#############################
# 3. RUN 
#############################

# Compression + indexation
bgzip -c $VCF > ${OUTPUTDIR}/${BASENAME}.vcf.gz
tabix -p vcf ${OUTPUTDIR}/${BASENAME}.vcf.gz

# Utilisation du vcf.gz pour les stats
VCFGZ=${OUTPUTDIR}/${BASENAME}.vcf.gz

# Global stats 
bcftools stats $VCFGZ > ${OUTPUTDIR}/${BASENAME}.global.stats

# Depth per individual 
vcftools --gzvcf $VCFGZ --depth --out ${OUTPUTDIR}/${BASENAME}.depth_indiv

# Missingness
vcftools --gzvcf $VCFGZ --missing-indv --out ${OUTPUTDIR}/${BASENAME}.missing_indiv

# Heterozygosity
vcftools --gzvcf $VCFGZ --het --out ${OUTPUTDIR}/${BASENAME}.het

# Relatedness2
vcftools --gzvcf $VCFGZ --relatedness2 --out ${OUTPUTDIR}/${BASENAME}.relatedness2

#### STATS SNP BCFTOOLS

In [60]:
import pandas as pd

data = [
    ("number of samples", 26),
    ("number of records", 33939150),
    ("number of no-ALTs", 33117278),
    ("number of SNPs", 796341),
    ("number of MNPs", 31198),
    ("number of indels", 177),
    ("number of others", 16),
    ("number of multiallelic sites", 7446),
    ("number of multiallelic SNP sites", 1028)
]

df = pd.DataFrame(data, columns=["Metric", "Value"])

styled_df = (
    df.style
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                     ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left')]}
    ])
    .format({"Value": lambda x: f"{x:,}".replace(",", " ")})  # séparation milliers
)

styled_df

df.to_csv("/Users/cibio/Desktop/Laurus/SDPOP/L_novocanariensis_L_azorica/STAT_VCF/bcftools_stat_L_novocanariensis_all.csv", index=False)

#### STATS **MISSING VALUES PER INDIVIDUALS**

In [61]:
import pandas as pd
from io import StringIO

data = """INDV	N_DATA	N_GENOTYPES_FILTERED	N_MISS	F_MISS
LN-T28_F	33939150	0	6989356	0.205938
LN-T27_F	33939150	0	6738427	0.198544
LN-T22_M	33939150	0	7562770	0.222833
LN-T20_M	33939150	0	7194982	0.211997
LN-T18b_M	33939150	0	5984425	0.176328
LN-T17b_M	33939150	0	6445765	0.189921
LN-T23_M	33939150	0	8079279	0.238052
LN-T16_M	33939150	0	7216216	0.212622
LN-T26_F	33939150	0	6374257	0.187814
LN-T24_M	33939150	0	6749369	0.198867
LN-T1_M	33939150	0	6696585	0.197312
LN-T2_F	33939150	0	7527307	0.221788
LN-T3_F	33939150	0	6215328	0.183132
LN-T14_M	33939150	0	6513999	0.191932
LN-T4_M	33939150	0	6291902	0.185388
LN-T11_M	33939150	0	7076134	0.208495
LN-T21_M	33939150	0	6873238	0.202517
LN-T5_F	33939150	0	6642154	0.195708
LN-T6_M	33939150	0	6204022	0.182798
LN-T7_M	33939150	0	6073564	0.178955
LN-T25	33939150	0	7183721	0.211665
LN-T19_M	33939150	0	6503096	0.19161
LN-T10_M	33939150	0	6689802	0.197112
LN-T15_M	33939150	0	8199363	0.24159
LN-T12_M	33939150	0	6864966	0.202273
LN-T13_F	33939150	0	7076390	0.208502"""

df = pd.read_csv(StringIO(data), sep="\t")

# Tri par missing décroissant
df = df.sort_values(by="F_MISS", ascending=False)

styled_df = (
    df.style
    .background_gradient(subset=["F_MISS"], cmap="Blues")
    .set_properties(**{
        'font-size': '11pt',
    })
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                    ('text-align', 'center'),]},
        {'selector': 'td', 'props': [('text-align', 'center')]}
    ])
    .format({
        "F_MISS": "{:.3f}",
        "N_MISS": lambda x: f"{x:,.0f}".replace(",", " "),  # sépare par milliers,
        "N_DATA": lambda x: f"{x:,.0f}".replace(",", " "),  # sépare par milliers,
    })
)

styled_df

df.to_csv("/Users/cibio/Desktop/Laurus/SDPOP/L_novocanariensis_L_azorica/STAT_VCF/Missing_stat_L_novocanariensis_all.csv", index=False)

#### STATS **MEAN DEPTH PER INDIVIDUALS**

In [62]:
import pandas as pd
from io import StringIO

# Données collées
# Données collées
data = """INDV	N_SITES	MEAN_DEPTH
LN-T28_F	26949794	304.858
LN-T27_F	27200723	313.878
LN-T22_M	26376380	297.323
LN-T20_M	26744168	262.643
LN-T18b_M	27954725	364.235
LN-T17b_M	27493385	370.031
LN-T23_M	25859871	195.867
LN-T16_M	26722934	215.566
LN-T26_F	27564893	330.321
LN-T24_M	27189781	328.221
LN-T1_M	27242565	273.248
LN-T2_F	26411843	217.181
LN-T3_F	27723822	386.186
LN-T14_M	27425151	332.484
LN-T4_M	27647248	341.323
LN-T11_M	26863016	336.057
LN-T21_M	27065912	348.167
LN-T5_F	27296996	316.947
LN-T6_M	27735128	331.132
LN-T7_M	27865586	389.073
LN-T25	26755429	235.459
LN-T19_M	27436054	323.93
LN-T10_M	27249348	313.947
LN-T15_M	25739787	278.575
LN-T12_M	27074184	313.403
LN-T13_F	26862760	283.063"""

df = pd.read_csv(pd.io.common.StringIO(data), sep="\t")
df = df.sort_values(by="MEAN_DEPTH", ascending=False)

styled_df = (
    df.style
    .background_gradient(subset=["MEAN_DEPTH"], cmap="Blues")
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                    ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center')]}
    ])
    .format({
        "N_SITES": lambda x: f"{x:,.0f}".replace(",", " "),  # sépare par milliers,
        "MEAN_DEPTH": "{:.1f}"
    })
)

styled_df

df.to_csv("/Users/cibio/Desktop/Laurus/SDPOP/L_novocanariensis_L_azorica/STAT_VCF/Mean_depth_stat_L_novocanariensis_all.csv", index=False)

# 2. SDPOP

## 2. 0 Before running SDpop Workflow

1. Create the file BED_EXON

In [ ]:
# creation BED_EXON
cd /home/tbertrand/work/L_novocanariensis/GENOME

grep "exon" braker.fixed_ids.gff3 \
| awk 'BEGIN{OFS="\t"}{
    match($9,/gene_id=([^;]+)/,a);
    print $1,$4-1,$5,a[1]
}' \
| sort -k1,1 -k2,2n \
> braker.fixed_ids_exons_sorted.bed

2. Select the individulals to filter from the vcf

In [ ]:
NONE

3. Adapt the rule pop_sum in order to choose corectly the individual sex depending on the indivudals name

In [ ]:
INDIV_LIST_FEMALES="LN-T28_F,LN-T27_F,LN-T26_F,LN-T2_F,LN-T3_F,LN-T5_F,LN-T13_F,LN-T4_M,LN-T19_M"
INDIV_LIST_MALES="LN-T22_M,LN-T20_M,LN-T18b_M,LN-T17b_M,LN-T23_M,LN-T16_M,LN-T24_M,LN-T1_M,LN-T14_M,LN-T11_M,LN-T21_M,LN-T6_M,LN-T7_M,LN-T10_M,LN-T15_M,LN-T12_M,LN-T25"

## 2.1 SDPOP WORKFLOW

#### config.yaml

In [ ]:
WORKDIR_SDPOP: "/home/tbertrand/work/L_novocanariensis/WORKFLOW_SD_POP"
VCF: "/home/tbertrand/work/L_novocanariensis/GENOTYPING/copy_vcf/test_concat.vcf"
ref_genome: "/home/tbertrand/work/L_novocanariensis/GENOME/Index/GCA_977007225.1_dmLauAzor1.pri_genomic.fna"
EXONS_BED: "/home/tbertrand/work/L_novocanariensis/GENOME/braker.fixed_ids_exons_sorted.bed"

VCF_NORM: "/home/tbertrand/work/L_novocanariensis/SD_POP/L_novocanariensis_Lazorica_norm.vcf"
VCF_NORM_FILT: "/home/tbertrand/work/L_novocanariensis/SD_POP/L_novocanariensis_Lazorica_norm_filtered.vcf"
VCF_NORM_FILT_EXONS_RECODE: "/home/tbertrand/work/L_novocanariensis/SD_POP/L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.vcf"
CNT_FILE: "/home/tbertrand/work/L_novocanariensis/SD_POP/geno_count_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out"

SDPOP_NOSEX_OUT: "/home/tbertrand/work/L_novocanariensis/SD_POP/SDPOP_nosexchr_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out"
SDPOP_XY_OUT: "/home/tbertrand/work/L_novocanariensis/SD_POP/SDPOP_XY_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out"
SDPOP_ZW_OUT: "/home/tbertrand/work/L_novocanariensis/SD_POP/SDPOP_ZW_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out"

NOSEX_LOG: "iteration_no_sex_chr_L_novocanariensis_Lazo.txt"
XY_LOG: "iteration_XY_sex_chr_L_novocanariensis_Lazo.txt"
ZW_LOG: "iteration_ZW_sex_chr_L_novocanariensis_Lazo.txt"

#### snakefile_SDpop

In [ ]:
# L objectif de ce script est prendre en INNPUT un fichier VCF brut (freebayes), de le formater pour SDpop et run SDpop

#######################################
# 1. CONFIG
#######################################

configfile: "config.yaml"

WORKDIR_SDPOP = config["WORKDIR_SDPOP"]
VCF           = config["VCF"]
ref_genome    = config["ref_genome"]
EXONS_BED     = config["EXONS_BED"]

VCF_NORM       = config["VCF_NORM"]
VCF_NORM_FILT  = config["VCF_NORM_FILT"]
VCF_NORM_FILT_EXONS_RECODE = config["VCF_NORM_FILT_EXONS_RECODE"]
CNT_FILE       = config["CNT_FILE"]

SDPOP_NOSEX_OUT = config["SDPOP_NOSEX_OUT"]
SDPOP_XY_OUT    = config["SDPOP_XY_OUT"]
SDPOP_ZW_OUT    = config["SDPOP_ZW_OUT"]

NOSEX_LOG = config["NOSEX_LOG"]
XY_LOG    = config["XY_LOG"]
ZW_LOG    = config["ZW_LOG"]


###########################################
# CIBLES FINALES 
###########################################

rule all:
    input:
        VCF_NORM,
        VCF_NORM_FILT,
        VCF_NORM_FILT_EXONS_RECODE,
        CNT_FILE,
        SDPOP_NOSEX_OUT,
        SDPOP_XY_OUT,
        SDPOP_ZW_OUT,
        NOSEX_LOG,
        XY_LOG,
        ZW_LOG


###############################################
# 3. RUN BCFTOOLS norm 
###############################################

rule bcftools_norm:
    input:
        vcf = VCF,
        ref = ref_genome
    output:
        temp(VCF_NORM)
    threads: 6
    resources:
        mem_mb=48000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/Bcftools/1.9

        bcftools norm --threads {threads} {input.vcf} -f {input.ref} -m +any -O v -o {output}
        """


##################################################
# 4. RUN FILTER VCFTOOLS 
##################################################
#Ici on filtre la profondeur minimal a 7 read par individus et on retire l individu Lm-F01Cb

rule vcftools_filter:
    input:
        VCF_NORM
    output:
        temp(VCF_NORM_FILT)
    threads: 1
    resources:
        mem_mb=8000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/samtools/1.14
        module load bioinfo/VCFtools/0.1.16

        vcftools --vcf {input} --minDP 7 --recode --out {output}
        mv {output}.recode.vcf {output}
        """


#########################################################
# 5. CUT EXONS WINDOWS 
#########################################################

rule exons_intersect:
    input:
        vcf = VCF_NORM_FILT,
        bed = EXONS_BED
    output:
        VCF_NORM_FILT_EXONS_RECODE
    threads: 1
    resources:
        mem_mb=8000,
        time="1-00:00:00"
    shell:
        """

        module load devel/python/Python-3.11.1
        module load bioinfo/bedtools/2.30.0

        # extract the header line (CHROM POS ...) of the vcf
        grep -m 1 "CHROM" "{input.vcf}" > "{output}"

        # create an intersect of the vcf and the bed file
        bedtools intersect -a "{input.vcf}" -b "{input.bed}" -wa -wb | \
        awk '{{
            ## We want to retrieve here the last column which corresponds to the name of the gene
            gene_id = $NF
            sub(/-.*/, "", gene_id)

            # print gene_id as the first column 
            printf "%s\t", gene_id

            # Print all the VCF column except the first (#CHROM) and the last 4 (BED column)
            for(i=2;i<=NF-4;i++) printf "%s\t", $i

            print ""
        }}' | sort -k1,1V -k2,2n | uniq >> "{output}"
        # The sort is in order to guarantees one continuous block per gene.
        """


#############################################
# 6. RUN POPSUM 
#############################################

rule popsum_counts:
    input:
        VCF_NORM_FILT_EXONS_RECODE
    output:
        CNT_FILE
    threads: 1
    resources:
        mem_mb=8000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148

        INDIV_LIST_FEMALES=$(grep "^#CHROM" "{input}" | cut -f10- | tr '\t' '\n' | grep '_F$' | tr '\n' ',')
        INDIV_LIST_MALES=$(grep "^#CHROM" "{input}" | cut -f10- | tr '\t' '\n' | grep '_M$' | tr '\n' ',')

        popsum "{input}" "{output}" v "$INDIV_LIST_FEMALES" "$INDIV_LIST_MALES"
        """


#############################################
# 7. RUN SDPOP 
#############################################

rule sdpop_nosex:
    input:
        CNT_FILE
    output:
        SDPOP_NOSEX_OUT
    log:
        NOSEX_LOG
    threads: 1
    resources:
        mem_mb=16000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148
        sdpop "{input}" "{output}" e n 1 1 s > "{log}"
        """

rule sdpop_xy:
    input:
        CNT_FILE
    output:
        SDPOP_XY_OUT
    log:
        XY_LOG
    threads: 1
    resources:
        mem_mb=16000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148
        sdpop "{input}" "{output}" e x 1 1 s > "{log}"
        """

rule sdpop_zw:
    input:
        CNT_FILE
    output:
        SDPOP_ZW_OUT
    log:
        ZW_LOG
    threads: 1
    resources:
        mem_mb=16000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148
        sdpop "{input}" "{output}" e z 1 1 s > "{log}"
        """

#### run_snakefile_SDpop.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=Snakemake_SDPOP_TEST_Lnobilis_L_azorica
#SBATCH -p workq
#SBATCH --time=2-00:00:00
#SBATCH --mail-user=tristan.bertrand@umontpellier.fr
#SBATCH --mail-type=ALL
#SBATCH --cpus-per-task=1
#SBATCH --mem=2G
#SBATCH --output=snakemake_%j.log

module load bioinfo/Snakemake/8.20.3

snakemake \
  --snakefile snakefile_test_SD_POP \
  --cores 6 \
  --rerun-incomplete \
  --keep-going \
  --latency-wait 60 \
  --executor slurm \
  --jobs 4 -n -p

## 2.2 SDPOP results

In [ ]:
# Import results 
scp tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_novocanariensis/SD_POP/SDPOP_XY_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_novocanariensis_L_azorica/SDPOP_resex
scp tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_novocanariensis/SD_POP/SDPOP_ZW_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_novocanariensis_L_azorica/SDPOP_resex
scp tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_novocanariensis/SD_POP/SDPOP_nosexchr_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_novocanariensis_L_azorica/SDPOP_resex


scp tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_novocanariensis/SD_POP/geno_count_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_novocanariensis_L_azorica/SDPOP_resex
scp tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_novocanariensis/GENOTYPING/STATS/L_novocanariensis_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf.gz /Users/cibio/Desktop/Laurus/SDPOP/L_novocanariensis_L_azorica/SDPOP_resex
scp tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_novocanariensis/GENOTYPING/STATS/L_novocanariensis_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf.gz.tbi /Users/cibio/Desktop/Laurus/SDPOP/L_novocanariensis_L_azorica/SDPOP_resex

In [ ]:
#L_novocanareinsis
grep '>' SDPOP_XY_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_XY_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_XY_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_ZW_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_ZW_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_ZW_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_nosexchr_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_nosexchr_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_nosexchr_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out.contig

### Results models SDPOP

XY
pi 0.872459  0.006900  0.094436  0.000365  0.025840 ; rho 0.493080  0.006920 ; e0: 5.016592e-04, log-likelihood: -5889812.929951, BIC (sites): 11779704.699718, BIC (contigs): 11779685.493029, BIC (sites*individuals): 11779723.422610

ZW
pi 0.893034  0.006962  0.099992  0.000000  0.000013 ; rho 0.500000  0.000000 ; e0: 6.202645e-04, log-likelihood: -5894670.528888, BIC (sites): 11789419.897594, BIC (contigs): 11789400.690905, BIC (sites*individuals): 11789438.620485

nosex
pi 0.893041  0.006962  0.099998 ; rho; e0: 6.205666e-04, log-likelihood: -5894670.653021, BIC (sites): 11789380.725951, BIC (contigs): 11789371.122607, BIC (sites*individuals): 11789390.087397

In [50]:
from IPython.display import display

# Données SDpop
import pandas as pd
import numpy as np

# Données SDpop
data = [
        {
        "Model": "XY",
        "autosomal.pr": 0.872459,
        "haploid.pr": 0.006900,
        "paralogs.pr": 0.094436,
        "hemizygote.pr": 0.000365,
        "sexlinked.pr": 0.025840,
        "rho": "0.493080, 0.006920",
        "e0": 5.016592e-04,
        "log_likelihood": -5889812.929951,
        "BIC_sites": 11779704.699718,
        "BIC_contigs": 11779685.493029,
        "BIC_sites_ind": 11779723.422610
    },
    {
        "Model": "ZW",
        "autosomal.pr": 0.893034,
        "haploid.pr": 0.006962,
        "paralogs.pr": 0.099992,
        "hemizygote.pr": 0.000000,
        "sexlinked.pr": 0.000013,
        "rho": "0.500000, 0.000000",
        "e0": 6.202645e-04,
        "log_likelihood": -5894670.528888,
        "BIC_sites": 11789419.897594,
        "BIC_contigs": 11789400.690905,
        "BIC_sites_ind": 11789438.620485
    },
    {
        "Model": "NOSEX",
        "autosomal.pr": 0.893041,
        "haploid.pr": 0.006962,
        "paralogs.pr": 0.099998,
        "hemizygote.pr": 0,
        "sexlinked.pr": 0,
        "rho": "",
        "e0": 6.205666e-04,
        "log_likelihood": -5894670.653021,
        "BIC_sites": 11789380.725951,
        "BIC_contigs": 11789371.122607,
        "BIC_sites_ind": 11789390.087397
    }
]

# Création du DataFrame
df = pd.DataFrame(data)

# Tri par log-likelihood décroissant pour choisir le meilleur modèle
df = df.sort_values(by="log_likelihood", ascending=False).reset_index(drop=True)

# Tableau 1 : statistiques de modèles
df_stats = df[[
    "Model",
    "log_likelihood",
    "BIC_sites",
    "BIC_contigs",
    "BIC_sites_ind"
]]

styled_stats = (
    df_stats.style
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                     ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left')]}
    ])
    .format({
        "log_likelihood": "{:,.3f}",
        "BIC_sites": "{:,.3f}",
        "BIC_contigs": "{:,.3f}",
        "BIC_sites_ind": "{:,.3f}"
    })
)

styled_stats

# Tableau 2 : priors
df_priors = df[[
    "Model",
    "autosomal.pr",
    "haploid.pr",
    "paralogs.pr",
    "hemizygote.pr",
    "sexlinked.pr"
]]

styled_priors = (
    df_priors.style
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                     ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left')]}
    ])
    .format({
    "autosomal.pr": "{:.6f}",
    "haploid.pr": "{:.6f}",
    "paralogs.pr": "{:.6f}",
    "hemizygote.pr": "{:.6f}",
    "sexlinked.pr": "{:.6f}"
})
)

styled_priors

# Afficher les deux tableaux
display(styled_stats)
display(styled_priors)

df_stats.to_csv("/Users/cibio/Desktop/Laurus/SDPOP/results_15_04/model_novocanariensis_stats.csv", index=False)
df_priors.to_csv("/Users/cibio/Desktop/Laurus/SDPOP/results_15_04/model_novocanariensis_priors.csv", index=False)

,Model,log_likelihood,BIC_sites,BIC_contigs,BIC_sites_ind
0,XY,"-5,889,812.930","11,779,704.700","11,779,685.493","11,779,723.423"
1,ZW,"-5,894,670.529","11,789,419.898","11,789,400.691","11,789,438.620"
2,NOSEX,"-5,894,670.653","11,789,380.726","11,789,371.123","11,789,390.087"


,Model,autosomal.pr,haploid.pr,paralogs.pr,hemizygote.pr,sexlinked.pr
0,XY,0.872459,0.006900,0.094436,0.000365,0.025840
1,ZW,0.893034,0.006962,0.099992,0.000000,0.000013
2,NOSEX,0.893041,0.006962,0.099998,0.000000,0.000000


## 2.4 Gametologs consensus sequence (**WZYZgenotyper**)

**script_WZYZ_genotyping.sh** 

In [ ]:
#!/bin/bash

#SBATCH --job-name=WZYZ_genotyping_novocanariensis
#SBATCH -p workq
#SBATCH --time=1-00:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=10G
#SBATCH --cpus-per-task=1

# This script is for the creation of consensus XY ZW haplotypes fro the genes infered as XY or ZW by SDPOP
# documentation: https://gitlab.in2p3.fr/sex-det-family/sdpop 

########################
# 0. MODULES
########################

module load bioinfo/SDpop/f112148

########################
# 1. INPUT 
########################

#path to the output file of SDPOP
SDPOP_FILE=/home/tbertrand/work/L_novocanariensis/SD_POP/SDPOP_XY_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out

#path to the gvcf/vcf file
GVCF=/home/tbertrand/work/L_novocanariensis/SD_POP/L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.vcf.gz
VCF=/home/tbertrand/work/L_novocanariensis/SD_POP/L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.vcf

#output path
OUTPUT=/home/tbertrand/work/L_novocanariensis/SD_POP/wXYz_allgenes_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out

########################
# 2. RUN
########################

gunzip -c "$GVCF" > "$VCF"

# v indicate wxyz_genotyper that we work on a vcf file
# wxyz_....out output file
# 0 : XY/ZW probability threshold: consensus sequences will be produced for any gene with a probability of being XY/ZW above the threshold (here 0)
# 0.95: alleles with a frequency above this threshold will be considered fixed (here 0.95)
# 0.6: alleles with a frequency above this threshold but below the threshold above will be considered not fixed and printed in lowercase letters.
# noprior_xy: work on no prior estimation of frequencies of x and y alleles
# XY: X and Y frequencies should be used
# mean: frequencies averaged over all subtypes should be used

wxyz_genotyper "$SDPOP_FILE" "$VCF" v "$OUTPUT" 0.8 0.95 0.6 noprior_xy XY mean

rm "$VCF"

In [ ]:
scp tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_novocanariensis/SD_POP/wXYz_allgenes_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_novocanariensis_L_azorica/SDPOP_resex

grep -E '^#|^>' wXYz_allgenes_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out > wXYz_allgenes_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out.contig

# ANNEXE: RERUN DES REGIONS AVEC TROP DE COUVERTURE

In [ ]:
genome_dir: "/home/tbertrand/work/L_novocanariensis/GENOME/Index"
ref_genome: "/home/tbertrand/work/L_novocanariensis/GENOME/Index/GCA_977007225.1_dmLauAzor1.pri_genomic.fna"
sample_table: "/home/tbertrand/work/L_novocanariensis/INPUT/reads_list_novocanariensis.txt"
mapping_dir: "/home/tbertrand/work/L_novocanariensis/MAPPING"
genotyping_dir: "/home/tbertrand/work/L_novocanariensis/GENOTYPING"

# Nouveau répertoire pour les VCF rerun
genotyping_dir_rerun: "/home/tbertrand/work/L_novocanariensis/GENOTYPING_rerun"

# Répertoire des BED existants
regions_dir_existing: "/home/tbertrand/work/L_novocanariensis/GENOTYPING/regions"

In [ ]:
configfile: "config_rerun.yaml"

# Lire table samples
samples = {}
with open(config["sample_table"]) as f:
    for line in f:
        r1, r2, ind = line.strip().split()
        samples[ind] = {"r1": r1, "r2": r2}

INDIVIDUALS = list(samples.keys())

regions_to_rerun = [115, 11605, 4691, 2599, 12559, 11652, 9298, 12451, 12464, 113, 11605]

# === Rule all ===
rule all:
    input:
        expand(config["genotyping_dir_rerun"] + "/vcf_chunks/variants.{i}.vcf", i=regions_to_rerun)

# === Variant calling pour ces régions ===
rule VariantCallingFreebayesRerun:
    input:
        bams=expand(config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam", ind=INDIVIDUALS),
        index=expand(config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam.bai", ind=INDIVIDUALS),
        ref=config["ref_genome"],
        samples=config["genotyping_dir"] + "/BAM_LIST.txt",
        # Utiliser les BED existants
        regions=lambda wildcards: f"{config['regions_dir_existing']}/region.{wildcards.i}.bed"
    output:
        temp(config["genotyping_dir_rerun"] + "/vcf_chunks/variants.{i}.vcf")
    log:
        config["genotyping_dir_rerun"] + "/logs/freebayes_region.{i}.log"
    threads: 1
    resources:
        mem_mb=300000,
        runtime=2880
    shell:
        """
        module load bioinfo/freebayes/1.3.6
        module load devel/Miniforge/Miniforge3
        module load bioinfo/vcflib/1.0.12

        mkdir -p {config[genotyping_dir_rerun]}/vcf_chunks
        mkdir -p {config[genotyping_dir_rerun]}/logs

        freebayes \
          -f {input.ref} \
          -t {input.regions} \
          -L {input.samples} \
          -C 2 \
          -g 20000 \
          --report-monomorphic \
          --use-best-n-alleles 2 \
          > {output} 2> {log}
        """


In [ ]:
#!/bin/bash
#SBATCH --job-name=Snakemake_L_novo_rerun
#SBATCH -p workq
#SBATCH --time=4-00:00:00
#SBATCH --mail-user=tristan.bertrand@umontpellier.fr
#SBATCH --mail-type=ALL
#SBATCH --cpus-per-task=1
#SBATCH --mem=8G
#SBATCH --output=snakemake_%j.log

module load bioinfo/Snakemake/8.20.3

snakemake \
  --snakefile snakefile_freebayes_rerun \
  --cores 12 \
  --rerun-incomplete \
  --keep-going \
  --latency-wait 60 \
  --executor slurm \
  --jobs 12 -n -p

In [ ]:
total 34M
-rw-r--r-- 1 tbertrand ISEM 3.2M Apr  8 20:44 variants.113.vcf
-rw-r--r-- 1 tbertrand ISEM 2.0M Apr  8 20:39 variants.115.vcf
-rw-r--r-- 1 tbertrand ISEM 1.4M Apr  8 20:41 variants.11605.vcf
-rw-r--r-- 1 tbertrand ISEM 3.5M Apr  8 20:44 variants.11652.vcf
-rw-r--r-- 1 tbertrand ISEM 2.0M Apr  8 20:41 variants.12451.vcf
-rw-r--r-- 1 tbertrand ISEM 1.9M Apr  8 20:44 variants.12464.vcf
-rw-r--r-- 1 tbertrand ISEM 4.1M Apr  8 20:44 variants.12559.vcf
-rw-r--r-- 1 tbertrand ISEM 2.1M Apr  8 20:39 variants.2599.vcf
-rw-r--r-- 1 tbertrand ISEM 2.1M Apr  8 20:44 variants.4691.vcf
-rw-r--r-- 1 tbertrand ISEM 2.2M Apr  8 20:42 variants.9298.vcf

In [ ]:
#!/bin/bash

set -euo pipefail

# ============================
# CONFIGURATION
# ============================

BAM_DIR="/path/to/mapping_dir"
BED_DIR="/path/to/regions_dir_existing"
REF="/path/to/reference.fa"
OUTPUT_DIR="coverage_analysis"

THREADS=4

# Liste des individus (adapte si besoin)
INDIVIDUALS=(ind1 ind2 ind3 ind4)

# Liste des régions à analyser
REGIONS=(115 11605 4691 2599 12559 11652 9298 12451 12464 113 11605)

# ============================
# INIT
# ============================

mkdir -p ${OUTPUT_DIR}/{depth,stats,mq30,duplicates,gc,logs}

echo "Starting coverage analysis..."

# ============================
# LOOP
# ============================

for region in "${REGIONS[@]}"; do

    BED_FILE="${BED_DIR}/region.${region}.bed"

    echo "Processing region ${region}"

    # GC content
    bedtools nuc -fi ${REF} -bed ${BED_FILE} \
        > ${OUTPUT_DIR}/gc/region.${region}.gc.txt

    for ind in "${INDIVIDUALS[@]}"; do

        BAM="${BAM_DIR}/${ind}_TranscriptomeGenomePos.sorted.rg.bam"

        PREFIX="${OUTPUT_DIR}/depth/${ind}.region.${region}"

        echo "  -> ${ind}"

        # ============================
        # RAW COVERAGE
        # ============================
        samtools depth -b ${BED_FILE} ${BAM} \
            > ${PREFIX}.depth.txt

        # ============================
        # MQ FILTERED COVERAGE
        # ============================
        samtools depth -Q 30 -b ${BED_FILE} ${BAM} \
            > ${OUTPUT_DIR}/mq30/${ind}.region.${region}.mq30.depth.txt

        # ============================
        # DUPLICATES
        # ============================
        samtools view -b -f 1024 ${BAM} | \
        samtools depth -b ${BED_FILE} - \
            > ${OUTPUT_DIR}/duplicates/${ind}.region.${region}.dup.depth.txt

        # ============================
        # STATS
        # ============================
        awk '{sum+=$3; n++; if($3>max) max=$3} 
             END {print "mean="sum/n, "max="max}' \
             ${PREFIX}.depth.txt \
             > ${OUTPUT_DIR}/stats/${ind}.region.${region}.stats.txt

    done
done

# ============================
# GLOBAL SUMMARY
# ============================

SUMMARY="${OUTPUT_DIR}/summary.tsv"
echo -e "individual\tregion\tmean_cov\tmax_cov" > ${SUMMARY}

for f in ${OUTPUT_DIR}/stats/*.stats.txt; do

    ind=$(basename $f | cut -d'.' -f1)
    region=$(basename $f | cut -d'.' -f3)

    mean=$(grep mean $f | awk -F'[= ]' '{print $2}')
    max=$(grep max $f | awk -F'[= ]' '{print $4}')

    echo -e "${ind}\t${region}\t${mean}\t${max}" >> ${SUMMARY}

done

echo "Analysis complete!"
echo "Summary file: ${SUMMARY}"